# Phase 4 — Revenue & LTV

**Questions:** what is a customer worth, and what does churn cost?

| Metric | Definition | Source |
|---|---|---|
| Revenue | Cash collected in the month of payment (NTD) | `mart_monthly_kpis` |
| ARPU | Monthly revenue ÷ month-end active subscribers | `mart_monthly_kpis` |
| Observed 12-month value | Money paid in a new customer's first 12 months (cohorts with 12 months observed) | `mart_ltv_by_segment` |
| Formula LTV | monthly price ÷ monthly churn rate — **assumes a constant churn rate** (monthly plans only) | `mart_value_at_risk` |
| Revenue lost to churn | Monthly-equivalent price of members who churned (label population) | `mart_value_at_risk` |

Currency is New Taiwan Dollars (NTD), as in the raw data. Annualised numbers are labelled **ASSUMPTION** (a churner would otherwise have paid 12 more months).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from streampulse.db import query, t, save_result, banner, RESULTS_DIR, IS_FIXTURE

banner()
FIG = RESULTS_DIR / "figures"
FIG.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})
MIN_SEGMENT_USERS = 5 if IS_FIXTURE else 1000

## 4.1 Revenue and ARPU over time

In [ ]:
kpi = query(f"select * from {t('mart_monthly_kpis')} order by month_start")
kpi["month_start"] = pd.to_datetime(kpi["month_start"])
k = kpi[~kpi.is_warmup_month].copy()

fig, ax1 = plt.subplots()
ax1.bar(k.month_start, k.revenue_ntd / 1e6, width=20, color="#9ecae1", label="Revenue (M NTD)")
ax2 = ax1.twinx()
ax2.plot(k.month_start, k.arpu_ntd, color="#08519c", marker="o", label="ARPU (NTD)")
ax1.set_ylabel("Revenue, million NTD"); ax2.set_ylabel("ARPU, NTD")
ax1.set_title("Monthly revenue and ARPU (warm-up months excluded)")
fig.savefig(FIG / "p04_revenue_arpu.png", dpi=150, bbox_inches="tight")
k[["month_start", "active_subscribers", "revenue_ntd", "arpu_ntd"]]

## 4.2 Observed 12-month value of new customers, by segment

In [ ]:
ltv = query(f"select * from {t('mart_ltv_by_segment')}")
ltv["quotable"] = ltv.customers >= MIN_SEGMENT_USERS
for dim in ltv.dimension.unique():
    display(ltv[ltv.dimension == dim].sort_values("avg_value_12m_ntd", ascending=False)
            .style.set_caption(dim).format({"customers": "{:,}", "avg_value_12m_ntd": "{:,.0f}",
                                            "total_value_12m_ntd": "{:,.0f}", "share_active_month_12": "{:.1%}"}))

## 4.3 Formula LTV for monthly plans
`LTV = monthly price / monthly churn`. Expected lifetime = 1 / churn months. It only holds if churn is roughly constant over a customer's life — Phase 3's retention curve tells you how wrong that is (churn is usually highest early).

In [ ]:
var = query(f"select * from {t('mart_value_at_risk')}")
monthly = (var[var.plan_length == "02. ~1 month"].groupby("auto_renew")[["users", "churners", "monthly_revenue_ntd"]].sum())
monthly["churn_rate"] = monthly.churners / monthly.users
monthly["avg_monthly_price_ntd"] = monthly.monthly_revenue_ntd / monthly.users
monthly["expected_lifetime_months_ASSUMPTION"] = 1 / monthly.churn_rate
monthly["formula_ltv_ntd_ASSUMPTION"] = monthly.avg_monthly_price_ntd / monthly.churn_rate
monthly

## 4.4 Revenue lost to churn and value at risk

In [ ]:
var = var.sort_values("monthly_revenue_lost_ntd", ascending=False)
total_lost = var.monthly_revenue_lost_ntd.sum()
total_rev = var.monthly_revenue_ntd.sum()
top3_share = var.head(3).monthly_revenue_lost_ntd.sum() / total_lost
print(f"Label population monthly revenue: {total_rev:,.0f} NTD")
print(f"Monthly revenue lost to churners: {total_lost:,.0f} NTD ({total_lost / total_rev:.1%} of the population's monthly revenue)")
print(f"ASSUMPTION x12 annualised: {total_lost * 12:,.0f} NTD")
print(f"Top 3 segments hold {top3_share:.1%} of revenue lost")
top = var.head(10)[["segment_label", "users", "churn_rate", "monthly_revenue_lost_ntd", "share_of_revenue_lost"]]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top.segment_label[::-1], top.monthly_revenue_lost_ntd[::-1] / 1e3, color="#de2d26")
ax.set_xlabel("Monthly revenue lost, thousand NTD"); ax.set_title("Top 10 segments by revenue lost to churn")
fig.savefig(FIG / "p04_value_at_risk.png", dpi=150, bbox_inches="tight")
top

In [ ]:
res = {
    "currency": "NTD",
    "months_used": [str(k.month_start.min().date()), str(k.month_start.max().date())],
    "revenue_ntd_last_full_month": float(k.revenue_ntd.iloc[-1]),
    "arpu_ntd": {"mean": float(k.arpu_ntd.mean()), "first": float(k.arpu_ntd.iloc[0]), "last": float(k.arpu_ntd.iloc[-1])},
    "observed_value_12m_by_first_plan": ltv[(ltv.dimension == "first_plan_group") & ltv.quotable]
        .set_index("segment")[["customers", "avg_value_12m_ntd", "share_active_month_12"]].to_dict("index"),
    "formula_ltv_monthly_plans_ASSUMPTION": monthly[["users", "churn_rate", "avg_monthly_price_ntd",
                                                      "formula_ltv_ntd_ASSUMPTION"]].to_dict("index"),
    "label_population_monthly_revenue_ntd": float(total_rev),
    "monthly_revenue_lost_ntd": float(total_lost),
    "share_of_monthly_revenue_lost": float(total_lost / total_rev),
    "annualised_revenue_lost_ntd_ASSUMPTION": float(total_lost * 12),
    "top3_segments_share_of_revenue_lost": float(top3_share),
    "top_segments_by_revenue_lost": var.head(5)[["segment_label", "users", "churn_rate",
                                                  "monthly_revenue_lost_ntd", "share_of_revenue_lost"]].to_dict("records"),
}
save_result("phase04_revenue_ltv", res)
res